# Notebook 1b — Finalize feature selection
Loads PSO/HHO subsets and writes `selected_features.json`. Attach the dataset with `fs_pso.json` + `fs_hho.json`.

In [1]:
# Cell 1 — Install
import subprocess, sys
def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', *args, '-q'], check=False)
pip('torch_geometric', 'numpy>=2')
pip('imbalanced-learn', 'numpy>=2')
print("Done. Restart kernel, then run Cell 2 onward.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 48.8 MB/s eta 0:00:00
Done. Restart kernel, then run Cell 2 onward.


In [2]:
# Cell 2 — Imports, seed, config. 
import os, gc, json, glob, random, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import torch, torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import coalesce
from gensim.models import Word2Vec
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.feature_selection import mutual_info_classif
from imblearn.over_sampling import SMOTE

GLOBAL_SEED = 42
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(s)
set_seed(GLOBAL_SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CONFIG = {
    'DATA_PATH'   : '/kaggle/input/datasets/monamehrun/pcos-cleaned-dataset/pcos_cleaned.csv',
    'TARGET_COL'  : 'PCOS',
    'OUTPUT_DIR'  : '/kaggle/working/',
    'TEST_SIZE'   : 0.20,
    'SEED'        : GLOBAL_SEED,
    'K_NEIGHBOURS': 10,
    'N2V_DIM'     : 64,
    'N2V_WALK_LEN': 20,
    'N2V_CONTEXT' : 10,
    'N2V_WALKS'   : 6,
    'N2V_EPOCHS'  : 20,
    'N2V_LR'      : 0.01,
    'HIDDEN_DIM'  : 64,
    'DROPOUT'     : 0.3,
    'LR'          : 0.001,
    'WEIGHT_DECAY': 1e-4,
    'EPOCHS'      : 90,
    'PATIENCE'    : 20,
}
SEARCH = {'N_FEATURES': None, 'N_FOLDS': 5, 'ALPHA': 0.90, 'MI_K': 20}
print("Config loaded. Device:", DEVICE)

Config loaded. Device: cuda


In [3]:
# Cell 3 — Load data
df = pd.read_csv(CONFIG['DATA_PATH'])
y = df[CONFIG['TARGET_COL']].values.astype(np.int64)
X = df.drop(columns=[CONFIG['TARGET_COL']]).values.astype(np.float32)
feature_names = df.drop(columns=[CONFIG['TARGET_COL']]).columns.tolist()

X_train, _Xte, y_train, _yte = train_test_split(
    X, y, test_size=CONFIG['TEST_SIZE'], stratify=y, random_state=CONFIG['SEED'])
del _Xte, _yte    # sealed — not used here

SEARCH['N_FEATURES'] = X_train.shape[1]
print(f"Training pool: {X_train.shape[0]} patients x {X_train.shape[1]} features "
      f"(PCOS+={int(y_train.sum())})")

Training pool: 432 patients x 48 features (PCOS+=141)


In [4]:
# Cell 4 — Pipeline utilities
def build_knn_graph(features_scaled, k=10):
    n = len(features_scaled)
    sim = cosine_similarity(features_scaled); np.fill_diagonal(sim, -2.0)
    src, dst, wts = [], [], []
    for i in range(n):
        for j in np.argpartition(sim[i], -k)[-k:]:
            w = float(max(0.0, sim[i][j])); src += [i, j]; dst += [j, i]; wts += [w, w]
    ei = torch.tensor([src, dst], dtype=torch.long); ew = torch.tensor(wts, dtype=torch.float)
    return coalesce(ei, ew, num_nodes=n, reduce='max')

def add_synthetic_nodes(ei, ew, X_real_sc, X_syn_sc, k=10):
    n_real, n_syn = len(X_real_sc), len(X_syn_sc)
    if n_syn == 0: return ei, ew
    sim = cosine_similarity(X_syn_sc, X_real_sc)
    src, dst, wts = [], [], []
    for i in range(n_syn):
        s = n_real + i
        for j in np.argpartition(sim[i], -k)[-k:]:
            w = float(max(0.0, sim[i][j])); src += [s, j]; dst += [j, s]; wts += [w, w]
    aug_ei = torch.cat([ei, torch.tensor([src, dst], dtype=torch.long)], dim=1)
    aug_ew = torch.cat([ew, torch.tensor(wts, dtype=torch.float)])
    return coalesce(aug_ei, aug_ew, num_nodes=n_real + n_syn, reduce='max')

def add_val_nodes(ei_aug, ew_aug, X_real_sc, X_val_sc, k=10, n_train_aug=None):
    n_real, n_val = len(X_real_sc), len(X_val_sc)
    if n_train_aug is None: n_train_aug = n_real
    sim = cosine_similarity(X_val_sc, X_real_sc)
    src, dst, wts = [], [], []
    for i in range(n_val):
        v = n_train_aug + i
        for j in np.argpartition(sim[i], -k)[-k:]:
            w = float(max(0.0, sim[i][j])); src += [v, j]; dst += [j, v]; wts += [w, w]
    comb_ei = torch.cat([ei_aug, torch.tensor([src, dst], dtype=torch.long)], dim=1)
    comb_ew = torch.cat([ew_aug, torch.tensor(wts, dtype=torch.float)])
    return comb_ei, comb_ew

def _random_walks(edge_index, num_nodes, walk_length, walks_per_node, seed=42):
    import random as _r; _r.seed(seed)
    adj = [[] for _ in range(num_nodes)]
    ei = edge_index.cpu().numpy()
    for s, d in zip(ei[0], ei[1]): adj[int(s)].append(int(d))
    walks, nodes = [], list(range(num_nodes))
    for _ in range(walks_per_node):
        _r.shuffle(nodes)
        for start in nodes:
            walk = [start]
            for _ in range(walk_length - 1):
                nbrs = adj[walk[-1]]
                if nbrs: walk.append(_r.choice(nbrs))
                else: break
            walks.append([str(n) for n in walk])
    return walks

def train_node2vec(edge_index, num_nodes, cfg):
    walks = _random_walks(edge_index, num_nodes, cfg['N2V_WALK_LEN'], cfg['N2V_WALKS'], cfg['SEED'])
    w2v = Word2Vec(sentences=walks, vector_size=cfg['N2V_DIM'], window=cfg['N2V_CONTEXT'],
                   min_count=0, sg=1, workers=1, seed=cfg['SEED'], epochs=cfg['N2V_EPOCHS'])
    emb = np.zeros((num_nodes, cfg['N2V_DIM']), dtype=np.float32)
    for idx in range(num_nodes):
        if str(idx) in w2v.wv: emb[idx] = w2v.wv[str(idx)]
    return emb

def inductive_n2v(X_new_sc, X_train_sc, n2v_train, k=10):
    sim = cosine_similarity(X_new_sc, X_train_sc)
    out = np.zeros((len(X_new_sc), n2v_train.shape[1]), dtype=np.float32)
    for i in range(len(X_new_sc)):
        tk = np.argpartition(sim[i], -k)[-k:]
        w = np.maximum(sim[i][tk], 0.0); ws = w.sum()
        w = w / ws if ws > 1e-9 else np.ones(k) / k
        out[i] = (n2v_train[tk] * w[:, None]).sum(axis=0)
    return out

class GraphSAGE(torch.nn.Module):
    def __init__(self, in_ch, hidden_ch, out_ch, dropout=0.3):
        super().__init__()
        self.c1 = SAGEConv(in_ch, hidden_ch); self.c2 = SAGEConv(hidden_ch, hidden_ch)
        self.lin = Linear(hidden_ch, out_ch); self.drop = dropout
    def forward(self, x, edge_index, edge_weight=None):
        x = F.relu(self.c1(x, edge_index)); x = F.dropout(x, self.drop, self.training)
        x = F.relu(self.c2(x, edge_index)); x = F.dropout(x, self.drop, self.training)
        return self.lin(x)

print("Pipeline utilities loaded.")

Pipeline utilities loaded.


In [5]:
# Cell 5 — Fitness. 5-fold CV MCC of GraphSAGE on the masked features (graph + Node2Vec
# rebuilt inside every fold => no leakage), combined with a parsimony penalty. 
def evaluate_subset(mask, X_pool, y_pool, cfg, n_folds):
    # returns (mean 5-fold val MCC, n_selected) using only the masked clinical columns
    cols = np.where(np.asarray(mask) == 1)[0]
    Xp = X_pool[:, cols]; n_clin = Xp.shape[1]; in_ch = n_clin + cfg['N2V_DIM']
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=cfg['SEED'])
    mccs = []
    for fold_idx, (tr, va) in enumerate(skf.split(Xp, y_pool)):
        try:
            X_tr, y_tr = Xp[tr], y_pool[tr]; X_va, y_va = Xp[va], y_pool[va]
            sc = StandardScaler(); X_tr_sc = sc.fit_transform(X_tr); X_va_sc = sc.transform(X_va)
            ei, ew = build_knn_graph(X_tr_sc, k=cfg['K_NEIGHBOURS']); n_real = len(X_tr_sc)
            n2v_tr = train_node2vec(ei, n_real, cfg)
            X_tr_full = np.concatenate([X_tr_sc, n2v_tr], axis=1)
            X_tr_sm, y_tr_sm = SMOTE(random_state=cfg['SEED'], k_neighbors=5).fit_resample(X_tr_full, y_tr)
            n_train_aug = len(X_tr_sm)
            if n_train_aug > n_real:
                ei, ew = add_synthetic_nodes(ei, ew, X_tr_sc, X_tr_sm[n_real:, :n_clin], k=cfg['K_NEIGHBOURS'])
            n2v_va = inductive_n2v(X_va_sc, X_tr_sc, n2v_tr, k=cfg['K_NEIGHBOURS'])
            X_va_full = np.concatenate([X_va_sc, n2v_va], axis=1)
            ei, ew = add_val_nodes(ei, ew, X_tr_sc, X_va_sc, k=cfg['K_NEIGHBOURS'], n_train_aug=n_train_aug)
            X_all = np.concatenate([X_tr_sm, X_va_full], axis=0); y_all = np.concatenate([y_tr_sm, y_va])
            tm = torch.zeros(len(X_all), dtype=torch.bool); tm[:n_train_aug] = True
            vm = torch.zeros(len(X_all), dtype=torch.bool); vm[n_train_aug:] = True
            data = Data(x=torch.tensor(X_all, dtype=torch.float), edge_index=ei, edge_weight=ew,
                        y=torch.tensor(y_all, dtype=torch.long), train_mask=tm, val_mask=vm).to(DEVICE)
            n_neg, n_pos = float((y_tr_sm == 0).sum()), float((y_tr_sm == 1).sum())
            crit = torch.nn.CrossEntropyLoss(
                weight=torch.tensor([1.0, n_neg / n_pos], dtype=torch.float).to(DEVICE))
            set_seed(cfg['SEED'] + fold_idx)
            model = GraphSAGE(in_ch, cfg['HIDDEN_DIM'], 2, cfg['DROPOUT']).to(DEVICE)
            opt = torch.optim.Adam(model.parameters(), lr=cfg['LR'], weight_decay=cfg['WEIGHT_DECAY'])
            sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=15, min_lr=1e-6)
            best_auroc, wait, best_w = 0.0, 0, None
            for _ in range(cfg['EPOCHS']):
                model.train(); opt.zero_grad()
                out = model(data.x, data.edge_index, data.edge_weight)
                loss = crit(out[data.train_mask], data.y[data.train_mask])
                if torch.isnan(loss): break
                loss.backward(); opt.step()
                model.eval()
                with torch.no_grad():
                    pv = torch.softmax(model(data.x, data.edge_index, data.edge_weight)[data.val_mask],
                                       dim=1)[:, 1].cpu().numpy()
                try: a = roc_auc_score(y_va, pv)
                except: a = 0.0
                sch.step(a)
                if a > best_auroc + 1e-4:
                    best_auroc, wait = a, 0
                    best_w = {k: v.clone() for k, v in model.state_dict().items()}
                else:
                    wait += 1
                    if wait >= cfg['PATIENCE']: break
            if best_w is not None: model.load_state_dict(best_w)
            model.eval()
            with torch.no_grad():
                pr = model(data.x, data.edge_index, data.edge_weight)[data.val_mask].argmax(dim=1).cpu().numpy()
            mccs.append(matthews_corrcoef(y_va, pr))
            del model, data, opt, sch, crit; gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        except Exception:
            mccs.append(-1.0)   # failed fold -> worst, never NaN
    return float(np.mean(mccs)), len(cols)

def make_fitness(X_pool, y_pool, cfg, search):
    cache = {}                                             # exact memoization: same mask -> same fitness
    def fitness(solution):
        mask = (np.asarray(solution) >= 0.5).astype(int)   # threshold continuous -> binary
        key = tuple(mask.tolist())
        if key in cache:                                   # skip re-evaluating a mask already seen
            return cache[key]
        n_sel = int(mask.sum())
        if n_sel < search['MIN_FEATURES']:
            cache[key] = -2.0; return -2.0                 # repel (we maximise)
        mcc, n = evaluate_subset(mask, X_pool, y_pool, cfg, search['N_FOLDS'])
        val = search['ALPHA'] * mcc - (1 - search['ALPHA']) * (n_sel / search['N_FEATURES'])
        cache[key] = val
        return val
    return fitness

print("Fitness ready.")

Fitness ready.


In [7]:
# Cell 6 — Load PSO/HHO from the dataset, compute ALL + MI baselines, write selected_features.json.
_base = '/kaggle/input/datasets/galibbhai/pso-hho'

pso = json.load(open(os.path.join(_base, 'fs_pso.json')))
hho = json.load(open(os.path.join(_base, 'fs_hho.json')))
print(f"Loaded  PSO {pso['n']} feats (MCC {pso['cv_mcc_5fold']:.4f}) | "
      f"HHO {hho['n']} feats (MCC {hho['cv_mcc_5fold']:.4f})")

all_mask = np.ones(SEARCH['N_FEATURES'], dtype=int)
all_mcc, all_n = evaluate_subset(all_mask, X_train, y_train, CONFIG, SEARCH['N_FOLDS'])
print(f"ALL -> {all_n} features | 5-fold MCC {all_mcc:.4f}")

mi = mutual_info_classif(X_train, y_train, random_state=GLOBAL_SEED)
mi_idx  = np.argsort(mi)[::-1][:SEARCH['MI_K']]
mi_mask = np.zeros(SEARCH['N_FEATURES'], dtype=int); mi_mask[mi_idx] = 1
mi_feats = [feature_names[i] for i in np.where(mi_mask == 1)[0]]
mi_mcc, mi_n = evaluate_subset(mi_mask, X_train, y_train, CONFIG, SEARCH['N_FOLDS'])
print(f"MI  -> {mi_n} features | 5-fold MCC {mi_mcc:.4f}")

results = {
    "PSO": {"features": pso['features'], "n": int(pso['n']), "cv_mcc_5fold": float(pso['cv_mcc_5fold'])},
    "HHO": {"features": hho['features'], "n": int(hho['n']), "cv_mcc_5fold": float(hho['cv_mcc_5fold'])},
    "ALL": {"features": feature_names, "n": int(all_n), "cv_mcc_5fold": float(all_mcc)},
    "MI":  {"features": mi_feats, "n": int(mi_n), "cv_mcc_5fold": float(mi_mcc)},
    "_meta": {"alpha": SEARCH['ALPHA'], "search_folds": SEARCH['N_FOLDS'], "mi_k": SEARCH['MI_K'],
              "seed": GLOBAL_SEED,
              "note": "5-fold MCC under the reduced search profile; final ranking is decided at "
                      "full settings + sealed test in later notebooks."},
}
print(f"\n{'Method':<6}{'#Feat':>7}{'5-fold MCC':>13}")
for k in ["PSO", "HHO", "MI", "ALL"]:
    print(f"{k:<6}{results[k]['n']:>7}{results[k]['cv_mcc_5fold']:>13.4f}")

out = os.path.join(CONFIG['OUTPUT_DIR'], 'selected_features.json')
with open(out, 'w') as f: json.dump(results, f, indent=2)
print(f"\nSaved -> {out}   (all four subsets carried forward)")

Loaded  PSO 22 feats (MCC 0.7878) | HHO 17 feats (MCC 0.7562)
ALL -> 48 features | 5-fold MCC 0.6769
MI  -> 20 features | 5-fold MCC 0.7525

Method  #Feat   5-fold MCC
PSO        22       0.7878
HHO        17       0.7562
MI         20       0.7525
ALL        48       0.6769

Saved -> /kaggle/working/selected_features.json   (all four subsets carried forward)
